## changing Directory

In [2]:
import os
%pwd

'c:\\Users\\Aswin\\MLprojects\\Quality_Of_wine(EndtoEnd ML)\\Quality_Of_Wine-EndtoEnd-\\research'

In [3]:
os.chdir("..")

In [4]:
%pwd

'c:\\Users\\Aswin\\MLprojects\\Quality_Of_wine(EndtoEnd ML)\\Quality_Of_Wine-EndtoEnd-'

## config entity

In [ ]:
# config entity

"""
An Entity is a simple Python class (usually a dataclass) that defines the structure of the data needed for this stage.
In this entity the hyperparams for the model training has to be filled too.

The Entity is a dataclass that defines exactly what data the training stage needs.
  What it contains: Paths to the transformed data, the path to save the final model, and 
                    hyperparameters (like alpha or l1_ratio).
         The "Why": It acts as a contract. It guarantees that the Trainer class will receive every
                    piece of information it needs in a structured way.

~WHY SPECIFCALLY USE THESE PARAMS FOR THE MODEL TRAINING?
  *In the .ipynb file do all the EDA and model training  with hyperpameter tuning then note that hyperparmas
   in params.yaml and in the entity to use it in the model training.

  *The model training is already done separeately to find the best params and best model for the project, then 
   it is noted and then put here for the model training.


"""
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path  
    model_name: str           # takes the model name from the config.yaml 
    alpha: float              # hyperparams
    l1_ratio: float           # hyperparams
    target_column: str

## Configuration Manager

In [7]:
from Quality_of_Wine.constants import *
from Quality_of_Wine.utils.common import read_yaml, create_directories

In [ ]:
# Configuration Manager

""" 
The Configuration Manager is the "Central Brain" that reads your config.yaml and params.yaml files.
   Its Role: It uses the read_yaml function from your utils/common.py to pull raw strings and 
             numbers from your configuration files.
   The Goal: It converts "text in a file" into a Python Entity object.


"""
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    """
    This is a specific method inside the ConfigurationManager class.
       Function: It fetches the "Model Trainer" section from your YAML files.
         Output: It returns a populated ModelTrainerConfig object.
            Why: This keeps your paths and parameters out of your training logic, so you
                 can change a learning rate in a YAML file without ever touching your Python code.

    """
    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.ElasticNet
        schema =  self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            train_data_path = config.train_data_path,
            test_data_path = config.test_data_path,
            model_name = config.model_name,
            alpha = params.alpha,
            l1_ratio = params.l1_ratio,
            target_column = schema.name
            
        )

        return model_trainer_config

## Model Trainer

In [11]:
import pandas as pd
import os
from Quality_of_Wine import logger
from sklearn.linear_model import ElasticNet
import joblib

In [14]:
# Model Trainer


class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    """ 
    This is the primary method inside the ModelTrainer class.
        Action: It calls .fit() on the data.
        Output: Once training is finished, it uses joblib or your save_bin utility 
                to save the trained model as a .joblib or .pickle file in the artifacts/ folder.
       Logging: It usually logs that the model was saved successfully and at what path.
    """
    def train(self):
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)


        train_x = train_data.drop([self.config.target_column], axis=1)
        test_x = test_data.drop([self.config.target_column], axis=1)
        train_y = train_data[[self.config.target_column]]
        test_y = test_data[[self.config.target_column]]


        lr = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio, random_state=42)
        lr.fit(train_x, train_y)

        joblib.dump(lr, os.path.join(self.config.root_dir, self.config.model_name))

## Model Trainer Pipeline

In [16]:
# Model Training Pipeline

""" 
The Pipeline is the orchestrator (the "Conveyor Belt").
~It calls the Configuration Manager.
~It requests the Model Trainer Config.
~It passes that config to the Model Trainer Class.
~It executes the train method.

"""
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config=model_trainer_config)
    model_trainer_config.train()
except Exception as e:
    raise e

[2026-03-05 13:28:58,348: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-05 13:28:58,348: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-05 13:28:58,358: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-03-05 13:28:58,358: INFO: common: created directory at: artifacts]
[2026-03-05 13:28:58,358: INFO: common: created directory at: artifacts/model_trainer]


## modular coding for model trainer in src folder

In [ ]:
"""
~Succcessfully ran the MODEL TRAINER part of the project in reseach folder.

~End To End MLOPS Data Science Project Implementation With Deployment(YT Channel)=https://youtu.be/pxk1Fr33-L4?si=YXioCQhqeFOWQ_az

~Now it has to be covered into modular coding.
   the process and where it has to copied is mentioned below

   *Step 01: entity/config_entity.py
            ~ In the entity/config_entity.py copy and paste the code below the entity of data tansformation
              code from config entity markdown above, just the class.
   
   *Step 02: src/config/configuration.py
            ~ In the src/config/configuration.py copy and paste the def function of the code from Configuration Manager markdown
              above.
            ~ the other package must be imported from src/entity.

   *Step 03: create a new file in src/components/model_trainer.py(new file)
            ~ In the src/components/model_trainer.py copy and paste the code from model trainer markdown above with packages
             and import needed packages.
   
   *Step 04: create a new file in src/pipelines/stage_04_model_trainer.py(new file)
            ~ Before copying the pipeline code, some code have to coded first that is already in src/pipelines/stage_04_model_trainer.ipynb.
            ~ Check the code to be written before.
   
   *Step 05: To test the pipeline from the above step the execution must be done from main.py
            ~ From the code from the src/pipelines/stage_04_model_trainer.ipynb copy and paste the code with the packages in the main.py.
            ~ Delete the artifacts folder.
            ~ And execute it in the terminal -python main.py.

"""